# 02 — Data Processing & Feature Engineering
Load raw data, clean it, encode categorical variables, and engineer new features to improve model performance.

**New features** are derived from EDA insights (`01_eda.ipynb`):
- `age_group` — captures non-linear age patterns (strongest correlation: 0.285)
- `has_balance` — binary flag; many zero-balance customers exist
- `tenure_group` — segments customer loyalty stages
- `active_products` — combines `active_member` × `products_number` (interaction signal)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# Load raw data
data = pd.read_csv("../data/raw/Bank-Customer-Churn-Prediction.csv")
print(f"Shape: {data.shape}")
data.dtypes

In [ ]:
# Quality checks: missing values and duplicates
print("Missing values:\n", data.isnull().sum())
print("\nDuplicates:", data.duplicated().sum())

In [ ]:
# Drop customer_id — not a predictive feature
data = data.drop(columns=['customer_id'])

In [ ]:
# Inspect categorical columns and their unique values
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)
for col in categorical_cols:
    print(f"  {col}: {data[col].unique()}")

In [ ]:
# One-hot encode categorical columns (drop first to avoid multicollinearity)
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded_arr = encoder.fit_transform(data[categorical_cols])
encoded_names = encoder.get_feature_names_out(categorical_cols)

data_encoded = (data
    .drop(columns=categorical_cols)
    .join(pd.DataFrame(encoded_arr, columns=encoded_names)))

print("All numeric:", (data_encoded.dtypes == object).sum() == 0)
print("Shape:", data_encoded.shape)

In [ ]:
# Age groups — captures non-linear age/churn relationship
data_encoded['age_group'] = pd.cut(
    data_encoded['age'],
    bins=[0, 30, 40, 50, 60, 100],
    labels=['young', 'middle', 'senior', 'old', 'very_old']
)
data_encoded = pd.get_dummies(data_encoded, columns=['age_group'], drop_first=True)

In [ ]:
# Has balance flag — many customers hold zero balance
data_encoded['has_balance'] = (data_encoded['balance'] > 0).astype(int)

In [ ]:
# Tenure groups — segments new / medium / loyal customers
data_encoded['tenure_group'] = pd.cut(
    data_encoded['tenure'],
    bins=[-1, 2, 5, 10],
    labels=['new', 'medium', 'loyal']
)
data_encoded = pd.get_dummies(data_encoded, columns=['tenure_group'], drop_first=True)

In [ ]:
# Active-products interaction — combines two weak signals into one
data_encoded['active_products'] = data_encoded['active_member'] * data_encoded['products_number']

In [ ]:
# Final shape and column list
print("Final shape:", data_encoded.shape)
print("Columns:", data_encoded.columns.tolist())

In [ ]:
# Save processed dataset
data_encoded.to_csv("../data/processed/churn_dataset_clean.csv", index=False)
print("Saved → ../data/processed/churn_dataset_clean.csv")

In [ ]:
# Export processed dataset for Power BI
X = data_encoded.drop(columns=['churn'])
y = data_encoded['churn']
powerbi_df = X.copy()
powerbi_df['churn'] = y.values
powerbi_df.to_csv("../results/data_processed_powerbi.csv", index=False)
print(f"Exported {powerbi_df.shape[0]} rows, {powerbi_df.shape[1]} columns")